# MiniTcl: implementación del comando `expr` con SLY

Este cuaderno transforma la idea del archivo `topdown.ipynb` en un componente utilizable por MiniTcl. En vez de escribir el parser manualmente, usamos **SLY** (`Lexer` + `Parser`), conservamos un **AST**, aplicamos el patrón **Visitor** y añadimos evaluación.

El subconjunto implementado acepta:

- números enteros, reales y booleanos `TRUE`/`FALSE`;
- variables MiniTcl escritas como `$nombre`;
- paréntesis y operadores unarios `+`, `-`, `!`;
- potencia `**`, multiplicación, división, módulo, suma y resta;
- comparaciones `<`, `<=`, `>`, `>=`, `==`, `!=`;
- operadores lógicos `&&` y `||`;
- las formas `expr 1 + 2` y `expr {1 + 2}`.

> Alcance: es un `expr` educativo compatible con MiniTcl, no una reproducción total del `expr` de Tcl/Tk.

In [3]:
# Ejecute esta celda una sola vez si SLY no está instalado.
!pip install -q sly

## 1. Arquitectura

El intérprete principal de MiniTcl reconoce el comando y entrega el texto que sigue a `expr`. A partir de allí este componente realiza cuatro pasos:

1. `ExprLexer` convierte caracteres en tokens.
2. `ExprParser` aplica la gramática y construye el AST.
3. `Evaluator` recorre el AST y obtiene un valor.
4. `MiniTclExpr` expone una interfaz simple para el intérprete.

La separación es importante: el lexer principal de MiniTcl **no debe intentar analizar internamente** toda la expresión. Solo debe preservar su texto y delegarlo a este módulo.

## 2. Nodos del AST

Una expresión como `1 + 2 * 3` no se evalúa durante el análisis sintáctico. Primero se representa como un árbol: `+(1, *(2, 3))`.

In [ ]:
from dataclasses import dataclass
from typing import Any, Mapping
from sly import Lexer, Parser


@dataclass(frozen=True)
class Node:
    pass


@dataclass(frozen=True)
class Number(Node):
    value: int | float


@dataclass(frozen=True)
class Boolean(Node):
    value: bool


@dataclass(frozen=True)
class Variable(Node):
    name: str


@dataclass(frozen=True)
class Unary(Node):
    operator: str
    operand: Node


@dataclass(frozen=True)
class Binary(Node):
    operator: str
    left: Node
    right: Node

## 3. Lexer de expresiones

El token `VARIABLE` incluye el signo `$`. Esto reproduce la sustitución de variables que MiniTcl necesita dentro de `expr`. Los operadores de dos caracteres se declaran antes que los de un carácter para evitar reconocer `<=` como `<` seguido de `=`.

In [ ]:
class ExprLexer(Lexer):
    tokens = {
        NUMBER, VARIABLE, TRUE, FALSE,
        PLUS, MINUS, POWER, TIMES, DIVIDE, MOD,
        LT, LE, GT, GE, EQ, NE,
        AND, OR, NOT,
    }

    literals = '()'
    ignore = ' \t'

    POWER = r'\*\*'
    LE = r'<='
    GE = r'>='
    EQ = r'=='
    NE = r'!='
    AND = r'&&'
    OR = r'\|\|'
    PLUS = r'\+'
    MINUS = r'-'
    TIMES = r'\*'
    DIVIDE = r'/'
    MOD = r'%'
    LT = r'<'
    GT = r'>'
    NOT = r'!'
    TRUE = r'(?i:true)\b'
    FALSE = r'(?i:false)\b'
    VARIABLE = r'\$[A-Za-z_][A-Za-z0-9_]*'

    @_(r'(?:\d+\.\d*|\.\d+)(?:[eE][+-]?\d+)?|\d+[eE][+-]?\d+|\d+')
    def NUMBER(self, t):
        text = t.value
        t.value = float(text) if any(c in text for c in '.eE') else int(text)
        return t

    @_(r'\n+')
    def newline(self, t):
        self.lineno += t.value.count('\n')

    def error(self, t):
        raise SyntaxError(
            f"Carácter ilegal {t.value[0]!r} en la posición {self.index}"
        )

In [ ]:
lexer = ExprLexer()
[(tok.type, tok.value) for tok in lexer.tokenize('$x + 2 * (3 <= 4)')]

[('VARIABLE', '$x'),
 ('PLUS', '+'),
 ('NUMBER', 2),
 ('TIMES', '*'),
 ('(', '('),
 ('NUMBER', 3),
 ('LE', '<='),
 ('NUMBER', 4),
 (')', ')')]

## 4. Gramática estratificada de `expr`

Implementaremos directamente la gramática de MiniTcl:

```ebnf
expresion ::= or
or ::= and | or "||" and
and ::= igualdad | and "&&" igualdad
igualdad ::= relacional | igualdad ("==" | "!=") relacional
relacional ::= aditiva | relacional ("<" | "<=" | ">" | ">=") aditiva
aditiva ::= multiplicativa | aditiva ("+" | "-") multiplicativa
multiplicativa ::= unaria | multiplicativa ("*" | "/" | "%") unaria
unaria ::= potencia | ("+" | "-" | "!") unaria
potencia ::= primaria | primaria "**" unaria
primaria ::= NUMBER | VAR | TRUE | FALSE | "(" expresion ")"
```

En Python no podemos llamar `or` y `and` a métodos porque son palabras reservadas. Por eso, en el código SLY se denominan `or_expr` y `and_expr`; las demás categorías conservan su nombre. La jerarquía queda codificada en la gramática y no depende de una tabla externa de precedencia.

In [ ]:
class ExprParser(Parser):
    tokens = ExprLexer.tokens

    # expresion ::= or
    @_('or_expr')
    def expression(self, p):
        return p.or_expr

    # or ::= and | or "||" and
    @_('and_expr')
    def or_expr(self, p):
        return p.and_expr

    @_('or_expr OR and_expr')
    def or_expr(self, p):
        return Binary(p.OR, p.or_expr, p.and_expr)

    # and ::= igualdad | and "&&" igualdad
    @_('equality')
    def and_expr(self, p):
        return p.equality

    @_('and_expr AND equality')
    def and_expr(self, p):
        return Binary(p.AND, p.and_expr, p.equality)

    # igualdad ::= relacional | igualdad ("==" | "!=") relacional
    @_('relational')
    def equality(self, p):
        return p.relational

    @_('equality EQ relational', 'equality NE relational')
    def equality(self, p):
        return Binary(p[1], p.equality, p.relational)

    # relacional ::= aditiva | relacional ("<" | "<=" | ">" | ">=") aditiva
    @_('additive')
    def relational(self, p):
        return p.additive

    @_('relational LT additive', 'relational LE additive',
       'relational GT additive', 'relational GE additive')
    def relational(self, p):
        return Binary(p[1], p.relational, p.additive)

    # aditiva ::= multiplicativa | aditiva ("+" | "-") multiplicativa
    @_('multiplicative')
    def additive(self, p):
        return p.multiplicative

    @_('additive PLUS multiplicative', 'additive MINUS multiplicative')
    def additive(self, p):
        return Binary(p[1], p.additive, p.multiplicative)

    # multiplicativa ::= unaria | multiplicativa ("*" | "/" | "%") unaria
    @_('unary')
    def multiplicative(self, p):
        return p.unary

    @_('multiplicative TIMES unary', 'multiplicative DIVIDE unary',
       'multiplicative MOD unary')
    def multiplicative(self, p):
        return Binary(p[1], p.multiplicative, p.unary)

    # unaria ::= potencia | ("+" | "-" | "!") unaria
    @_('power')
    def unary(self, p):
        return p.power

    @_('PLUS unary', 'MINUS unary', 'NOT unary')
    def unary(self, p):
        return Unary(p[0], p.unary)

    # potencia ::= primaria | primaria "**" unaria
    @_('primary')
    def power(self, p):
        return p.primary

    @_('primary POWER unary')
    def power(self, p):
        return Binary(p.POWER, p.primary, p.unary)

    # primaria ::= NUMBER | VAR | TRUE | FALSE | "(" expresion ")"
    @_('NUMBER')
    def primary(self, p):
        return Number(p.NUMBER)

    @_('VARIABLE')
    def primary(self, p):
        return Variable(p.VARIABLE[1:])

    @_('TRUE')
    def primary(self, p):
        return Boolean(True)

    @_('FALSE')
    def primary(self, p):
        return Boolean(False)

    @_('"(" expression ")"')
    def primary(self, p):
        return p.expression

    def error(self, token):
        if token is None:
            raise SyntaxError('Expresión incompleta')
        raise SyntaxError(
            f"Token inesperado {token.value!r} en la línea {token.lineno}"
        )

## 5. Evaluador (Visitor)

En MiniTcl, los resultados lógicos se representan como `1` y `0`. `&&` y `||` usan evaluación de cortocircuito: el operando derecho solo se evalúa cuando hace falta.

In [ ]:
class ExprError(Exception):
    'Error controlado producido por expr.'


class Evaluator:
    def __init__(self, variables: Mapping[str, Any] | None = None):
        self.variables = dict(variables or {})

    def visit(self, node: Node):
        method = getattr(self, f'visit_{type(node).__name__}', None)
        if method is None:
            raise ExprError(f'No se puede evaluar {type(node).__name__}')
        return method(node)

    def visit_Number(self, node):
        return node.value

    def visit_Boolean(self, node):
        return int(node.value)

    def visit_Variable(self, node):
        if node.name not in self.variables:
            raise ExprError(f'Variable no definida: ${node.name}')
        value = self.variables[node.name]
        try:
            text = str(value)
            return float(text) if any(c in text for c in '.eE') else int(text)
        except (TypeError, ValueError):
            raise ExprError(f'El valor de ${node.name} no es numérico: {value!r}')

    def visit_Unary(self, node):
        value = self.visit(node.operand)
        operations = {
            '+': lambda x: +x,
            '-': lambda x: -x,
            '!': lambda x: int(not bool(x)),
        }
        return operations[node.operator](value)

    def visit_Binary(self, node):
        left = self.visit(node.left)

        # Cortocircuito: evita evaluar innecesariamente el lado derecho.
        if node.operator == '&&':
            return int(bool(left) and bool(self.visit(node.right)))
        if node.operator == '||':
            return int(bool(left) or bool(self.visit(node.right)))

        right = self.visit(node.right)
        operations = {
            '+':  lambda a, b: a + b,
            '-':  lambda a, b: a - b,
            '*':  lambda a, b: a * b,
            '/':  lambda a, b: a / b,
            '%':  lambda a, b: a % b,
            '**': lambda a, b: a ** b,
            '<':  lambda a, b: int(a < b),
            '<=': lambda a, b: int(a <= b),
            '>':  lambda a, b: int(a > b),
            '>=': lambda a, b: int(a >= b),
            '==': lambda a, b: int(a == b),
            '!=': lambda a, b: int(a != b),
        }
        try:
            return operations[node.operator](left, right)
        except ZeroDivisionError as exc:
            raise ExprError('División por cero') from exc

## 6. Fachada: el comando `expr`

La clase siguiente une lexer, parser y evaluador. También retira, de forma controlada, las llaves externas que MiniTcl usa para agrupar la expresión.

Ejemplos válidos:

```tcl
expr 1 + 2 * 3
expr {1 + 2 * 3}
set x 10
expr {$x >= 10 && $x < 20}
```

In [ ]:
class MiniTclExpr:
    def __init__(self):
        self.lexer = ExprLexer()

    @staticmethod
    def _unwrap(text: str) -> str:
        text = text.strip()
        if text.startswith('{') or text.endswith('}'):
            if not (text.startswith('{') and text.endswith('}')):
                raise ExprError('Llaves exteriores no balanceadas')
            text = text[1:-1].strip()
        if not text:
            raise ExprError('expr requiere una expresión')
        return text

    def ast(self, text: str) -> Node:
        source = self._unwrap(text)
        # Un parser nuevo evita conservar estado después de un error.
        return ExprParser().parse(self.lexer.tokenize(source))

    def evaluate(self, text: str, variables=None):
        tree = self.ast(text)
        return Evaluator(variables).visit(tree)


expr = MiniTclExpr()
expr.evaluate('{1 + 2 * 3}')

In [ ]:
variables = {'x': 10, 'y': 3.5, 'cero': 0}

examples = [
    '1 + 2 * 3',
    '(1 + 2) * 3',
    '-$x + 4',
    '$x >= 10 && $y < 4',
    '$cero && (1 / 0)',       # demuestra el cortocircuito
    '{2 + 3 * 4 == 14}',
]

for source in examples:
    print(f'{source:25} -> {expr.evaluate(source, variables)}')

## 7. Visualización textual del AST

Esta versión no depende de Graphviz, de modo que funciona tanto en Jupyter Notebook como en Google Colab. La sangría muestra la estructura y permite comprobar la precedencia.

In [ ]:
class TreePrinter:
    def render(self, node: Node, prefix='', last=True):
        connector = '└── ' if last else '├── '
        label = self._label(node)
        lines = [prefix + connector + label]
        children = self._children(node)
        child_prefix = prefix + ('    ' if last else '│   ')
        for index, child in enumerate(children):
            lines.extend(self.render(child, child_prefix, index == len(children) - 1))
        return lines

    @staticmethod
    def _label(node):
        if isinstance(node, Number):
            return f'NUMBER {node.value}'
        if isinstance(node, Boolean):
            return f'BOOLEAN {str(node.value).upper()}'
        if isinstance(node, Variable):
            return f'VARIABLE ${node.name}'
        if isinstance(node, Unary):
            return f'UNARY {node.operator}'
        return f'BINARY {node.operator}'

    @staticmethod
    def _children(node):
        if isinstance(node, Unary):
            return [node.operand]
        if isinstance(node, Binary):
            return [node.left, node.right]
        return []

    def __call__(self, node):
        return '\n'.join(self.render(node))


tree = expr.ast('$x + 2 * (3 - 1)')
print(TreePrinter()(tree))

## 8. Integración con el intérprete MiniTcl

El despachador de comandos puede conservar una tabla de variables y delegar en `MiniTclExpr`:

In [ ]:
class MiniTclDemo:
    'Integración mínima; no reemplaza el intérprete completo de MiniTcl.'

    def __init__(self):
        self.variables = {}
        self.expr_engine = MiniTclExpr()

    def execute(self, command: str):
        command = command.strip()
        if command.startswith('set '):
            _, name, value = command.split(maxsplit=2)
            self.variables[name] = value
            return value
        if command.startswith('expr '):
            expression_text = command[5:]
            return self.expr_engine.evaluate(expression_text, self.variables)
        raise ExprError('Comando desconocido')


mini = MiniTclDemo()
print(mini.execute('set x 8'))
print(mini.execute('expr {$x * 2 + 1}'))

## 9. Pruebas automáticas

Estas pruebas verifican precedencia, asociatividad, paréntesis, variables, comparaciones, lógica y errores.

In [ ]:
tests = [
    ('1 + 2 * 3', {}, 7),
    ('(1 + 2) * 3', {}, 9),
    ('10 - 3 - 2', {}, 5),
    ('-2 * 3', {}, -6),
    ('10 % 4', {}, 2),
    ('2 ** 3 ** 2', {}, 512),
    ('-2 ** 2', {}, -4),
    ('2 ** -2', {}, 0.25),
    ('$a + 1', {'a': 9}, 10),
    ('3 < 4', {}, 1),
    ('3 == 4', {}, 0),
    ('1 || 0 && 0', {}, 1),
    ('!0', {}, 1),
    ('TRUE && FALSE', {}, 0),
    ('TRUE || FALSE', {}, 1),
]

for source, variables, expected in tests:
    actual = expr.evaluate(source, variables)
    assert actual == expected, (source, expected, actual)

error_tests = [
    ('1 / 0', {}, 'División por cero'),
    ('$ausente + 1', {}, 'Variable no definida'),
    ('1 +', {}, 'Expresión incompleta'),
]

for source, variables, expected_message in error_tests:
    try:
        expr.evaluate(source, variables)
        raise AssertionError(f'Se esperaba error en {source!r}')
    except (ExprError, SyntaxError) as error:
        assert expected_message in str(error), (source, error)

print(f'{len(tests) + len(error_tests)} pruebas superadas')

## 10. Ejercicios propuestos

1. Extienda el evaluador con funciones `abs(x)`, `min(a,b)` y `max(a,b)`.
2. Haga que la división entre dos enteros tenga la semántica que haya definido formalmente para MiniTcl.
3. Integre `MiniTclExpr` con el comando `if` del intérprete.
4. Compare los errores producidos por este subconjunto con los de `tclsh`.
5. Escriba pruebas que demuestren por qué `2 ** 3 ** 2` es asociativa por la derecha.
6. Añada una fase de análisis semántico separada de la evaluación.

### Decisión de diseño para el proyecto

Use dos niveles:

- **Lexer principal de MiniTcl:** separa comandos y argumentos, respetando llaves y comillas.
- **`ExprLexer` + `ExprParser`:** reciben únicamente el argumento de `expr` y procesan su lenguaje interno.

SLY puede emplearse en ambos niveles, pero deben ser clases independientes porque reconocen lenguajes y contextos distintos.